<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Lab: Structure Unstructured Restaurant Data with an LLM**


Estimated time needed: **45** minutes


## **Scenario**


### Background

You are a Data Engineer at a leading AI startup building a next-generation *Connoisseur Companion*. Unlike traditional recommendation systems that rely on coarse 1–5 star ratings, the organization aims to understand the **why** behind user preferences, capturing the vibe of a restaurant, dietary considerations, and standout **“hero dishes”** hidden inside unstructured, multimodal data.

### The challenge

The organization has acquired a large, messy dataset spanning multiple modalities:

* **Restaurant descriptions** and **user reviews** (raw TXT)
* **Food recipes** with structured metadata and images (JSON + JPEG)
* **User restaurant visit histories** (JSON with URLs)

However, this data is not immediately usable:

1. **Unstructured text** lacks a consistent schema, making search and retrieval inefficient.
2. **Images and URLs** are opaque to traditional data pipelines and must be transformed into searchable, quantitative representations.
3. The data spans multiple modalities with no unified representation.

To build a high-quality, explainable recommendation engine, your first task is to transform this **heterogeneous, multimodal data** into a **structured knowledge base**. This includes extracting semantic signals from **text**, generating representations for **images** and **URLs**, and organizing everything into a coherent, query-friendly JSON format.


## **Objectives**


In this lab, you will write a Python program that will:

* Load raw restaurant descriptions and review data from unstructured TXT files
* Use a multimodal/LLM-based pipeline to extract structured attributes (e.g., cuisine type, ambiance, dietary options, and signature dishes)
* Convert the extracted information into a well-defined JSON schema suitable for indexing and search


## **Important: About the lab environment**


Please be aware that sessions for this lab environment are not persisted. Every time you connect to this lab, a new environment is created for you. Any data you may have saved in the earlier session would get lost. Plan to complete these labs in a single session, to avoid losing your data.


## **Screenshot requirement for this lab**


You will be prompted to take a screenshot and save it on your own device. You will need this screenshot either to answer graded quiz questions or to upload as your submission for the Final Project at the end of this course. You can use various free screen-grabbing tools or your operating system's shortcut keys to do this (for example, `Alt+PrintScreen` on Windows and `Command+shift+4` on Mac).

**Note**: The screenshot can be saved with either the **.jpg** or **.png** extension.


----


## **Set up the lab environment**


For this lab, you will be using the following libraries:

* [`numpy`](https://numpy.org/) for numerical operations and handling array-based data during preprocessing and analysis

* [`matplotlib`](https://matplotlib.org/) for basic data visualization and plotting, useful for inspecting distributions or intermediate results

* [`json`](https://docs.python.org/3/library/json.html) for parsing, constructing, and serializing structured JSON representations extracted from unstructured text

* **IBM Watsonx AI SDK** (`ibm-watsonx-ai`) for interacting with foundation models hosted on IBM watsonx.ai:

  * `Credentials` to securely authenticate with the watsonx.ai service
  * `ModelInference` to invoke foundation models for text understanding and information extraction
  * `GenTextParamsMetaNames` to configure text generation and extraction parameters
  * `ModelTypes` and `DecodingMethods` to select the appropriate model and control inference behavior

These libraries together enable you to transform raw restaurant description text into structured, machine-readable knowledge using GenAI-powered workflows.


### Install required libraries

Run the following code block to install all required libraries:


In [1]:
%%capture
%pip install numpy==2.3.4
%pip install matplotlib==3.10.7
%pip install ibm-watsonx-ai==1.4.7

### Import required libraries

It is recommended to import all required libraries in one place (here):


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import json
import os

# IBM WatsonX imports
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import (
    ModelTypes,
    DecodingMethods,
)

# Libraries and codes to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

### Fetch the data file


Run the following code to fetch the restaurant description text file:


In [3]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/1r_mM6ZPYNxcFv65QkzubA/California-Culinary-Map.txt

--2026-11-22 17:08:26--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/1r_mM6ZPYNxcFv65QkzubA/California-Culinary-Map.txt
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 74679 (73K) [text/plain]
Saving to: ‘California-Culinary-Map.txt.3’

California-Culinary 100%[===================>]  72.93K  --.-KB/s    in 0.002s  

2026-11-22 17:08:26 (31.1 MB/s) - ‘California-Culinary-Map.txt.3’ saved [74679/74679]



----


## **Exercise 1: Load the data and display the texts**


Great! You now have the **California-Culinary-Map.txt** file in your folder. Before diving into the data, it is important to load and explore it first to gain a high-level understanding of its structure and content. This exercise gives you the opportunity to inspect the raw data and prepare it for transformation into a more structured, machine-accessible format.


### Step 1: Load the data

In Step 1, you will load the text data file and explore the restaurant contents. Make sure you use the correct file path that links to the file California-Culinary-Map.txt. 


In [4]:
### Your Code Here:
### 1.1: Define the file_path to the text file
file_path =  "California-Culinary-Map.txt"

### 1.2: Open the text file
with open(file_path, 'r') as file:
    data = file.read()

### 1.3: Print the first 100 characters of the restaurant data
print(data[:100])

### The Culinary Map of California

**The Gilded Artichoke** brings a **bohemian chic** energy to th


### Step 2: Split the restaurant paragraphs into a Python list

In the previous step, you may have noticed that the text file consists of multiple paragraphs, each describing a single restaurant. In Step 2, you will split these paragraphs into a list, allowing you to work with and manage each restaurant’s data more effectively.


In [12]:
### Your Code Here:
### 2.1: Split the restaurant paragraphs into list (hint: use .split('\n\n')
restaurant_list = data.split('\n\n')

### 2.2: Since the first item is the dataset name, we remove it
restaurant_list = restaurant_list[1:] #Delete first row in list 

### 2.3: Print out the number of restaurants we have (hint: use the len() function)
print(len(restaurant_list))

### 2.4: Print out the first item to have a closer look at the content
print(restaurant_list[1])

210
Down in **Santa Monica**, **Mar de Cortez** serves as a **sun-drenched**, **casual taqueria** specializing in **Baja-style seafood**. With a **4.2/5** rating, it captures the salt-air energy of the coast through its signature beer-battered snapper tacos and zesty octopus ceviche, making it a premier spot for open-air dining near the pier. Price range: $


----


## **Exercise 2: Define the LLM**


You may have observed that each restaurant description includes the following key attributes: 
1. Restaurant name
2. Location
3. Restaurant type
4. Food style
5. Rating
6. Price range
7. Signature dishes
8. Specialties
9. Shortcomings

This information is essential for uniquely identifying and describing a restaurant.

In this lab, you will use a large language model (LLM) to organize these attributes into a structured JSON format. As a first step, you will define the base LLM that will extract and structure the information from the unstructured text.


### Step 1: Implement the LLM function


You will use **IBM Granite 4H Small** as the base LLM for this lab, as it provides strong text understanding capabilities while remaining cost-efficient:


In [13]:
def llm_model(system_msg, prompt_txt):
    #system_msg: the system message given to the LLM
    #prompt_txt: the user prompt
    
    model_id = "ibm/granite-4-h-small"

    project_id="skills-network"

    credentials = Credentials(url = "https://us-south.ml.cloud.ibm.com")

    ### 1.1: Define the model by ModelInference
    model = ModelInference(
        model_id=model_id,
        credentials=credentials,
        project_id=project_id
    )

    ### 1.2: Define the messages
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": prompt_txt}
    ]

    ### 1.3: Get the final response output and return it
    response = model.chat(messages=messages)
    output_text = response["choices"][0]["message"]["content"]
    
    return output_text

### Step 2: Test your llm_model()


Test your LLM function with the following inputs:


In [14]:
import time

def safe_llm_call(system_msg, prompt_txt, retries=3):
    for i in range(retries):
        try:
            return llm_model(system_msg, prompt_txt)
        except Exception:
            time.sleep(2)
    return "Failed after retries"

In [15]:
system_msg = "You are a helpful assistant."
prompt_txt = "Which place is warmer in winter? Hawaii or Greenland?"
print(safe_llm_call(system_msg, prompt_txt))

Hawaii is significantly warmer in winter compared to Greenland.

Hawaii:
- Hawaii's tropical location and surrounding ocean waters moderate temperatures.
- In winter (December-February), average high temperatures typically range from the mid-70s to low 80s °F (24-28 °C).
- Lows usually stay in the 60s °F (15-21 °C).
- Even the cooler winter months in Hawaii are relatively warm.

Greenland:
- Greenland is an Arctic island located in the far northern part of North America.
- Winters in Greenland are long, cold, and dark.
- Even in the "warmest" winter month (July, which is summer in the northern hemisphere), temperatures average below freezing (0 °C or 32 °F). 
- Typical winter highs in December-February range from -4°C to -15°C (25°F to 5°F).
- Lows often drop well below freezing, regularly below -20°C (-4°F).

In summary, Hawaii has a tropical climate with warm temperatures year-round, including winter, while Greenland has a polar climate with very cold winter temperatures. If you're l

----


## **Exercise 3: Prompt Engineering**


To make an LLM behave reliably and produce outputs that meet your requirements, careful prompt design is essential. Here is a recap of key prompt engineering techniques:

* **Clear and specific instructions**: Explicitly state the task, expected output format, and constraints to reduce ambiguity.
* **Structured output guidance**: Provide schemas, templates, or examples (e.g., JSON formats) to encourage consistent, machine-readable responses.
* **Few-shot prompting**: Include representative input–output examples to guide the model toward the desired behavior.
* **Role prompting**: Assign the model a specific role (e.g., “You are a data extraction assistant”) to shape its reasoning and tone.
* **Constraint-based prompting**: Define what the model should and should not do to improve precision and reliability.
* **Iterative refinement**: Evaluate outputs and progressively refine prompts to improve performance over time.

This exercise focuses on one-shot prompting, a special case of few-shot prompting. You will define a function that generates the prompt templates.


### Step 1: Define the template


In **Step 1**, you will implement a function that generates prompts for the LLM, given an input restaurant description paragraph.

You will apply the **one-shot prompting** technique, which includes a single example of the desired output within the prompt to guide the model’s response.

An example output is provided, corresponding to the second restaurant paragraph in your `restaurant_list` variable. Using this example, you will design a prompt template that enables the model to consistently transform *any* input restaurant description into the required structured JSON format.

**Note:** For the price range field, instruct the LLM to convert dollar signs (e.g., \\\\\\\\\\\\\\\\\\\\\\\\$, \\$\\$, $$$) into an integer representing the number of dollar symbols.


In [17]:
print(restaurant_list)
EXAMPLE_RESTAURANT_PARAGRAPH = restaurant_list
print(restaurant_list[1])

['**The Gilded Artichoke** brings a **bohemian chic** energy to the hills of **Silver Lake**, operating as an **upscale bistro** that prioritizes **Farm-to-Table Californian** ingredients. The space feels like a high-end greenhouse with its reclaimed wood and floor-to-ceiling windows, perfectly complementing the **4.5/5** rating earned by its lavender-rubbed roasted chicken and delicate heirloom tomato tarts.  Price range: $$$$', 'Down in **Santa Monica**, **Mar de Cortez** serves as a **sun-drenched**, **casual taqueria** specializing in **Baja-style seafood**. With a **4.2/5** rating, it captures the salt-air energy of the coast through its signature beer-battered snapper tacos and zesty octopus ceviche, making it a premier spot for open-air dining near the pier. Price range: $', '**Iron & Embers** anchors the **Arts District** in **DTLA** as a **moody**, **industrial American steakhouse**. This **fine dining** establishment earns a **4.8/5** for its commitment to the flame, featurin

In [18]:
EXAMPLE_RESTAURANT_PARAGRAPH = restaurant_list[1] #use the second restaurant paragraph as the example
EXAMPLE_OUTPUT = """
    {{
    "name": "Mar de Cortez",
    "location": "Santa Monica",
    "type": "casual taqueria",
    "food_style": "Baja-style seafood",
    "rating": 4.2,
    "price_range": 1,
    "signatures": [
        "beer-battered snapper tacos",
        "zesty octopus ceviche"
    ],
    "vibe": "salt-air energy",
    "environment": "a premier sun-drenched spot for open-air dining near the pier."
    "shortcomings": []
    }}
"""

### Design your prompt here
def restaurant_data_structure_prompt_generation(restaurant_paragraph):
    base_system_msg = f"""
    You are an expert restaurant data extraction assistant.
    Extract restaurant information accurately.
    Return valid JSON only.
    Do not invent missing information.
    Follow the required JSON structure
    """
    
    base_user_prompt = f"""
    Task:
     Extract the restaurant information from the description
     and convert it into the JSON format shown in the example.

    Restaurant description:
    {restaurant_paragraph}

    Example:
    Input Restaurant Description: {EXAMPLE_RESTAURANT_PARAGRAPH}
    Output:
    {EXAMPLE_OUTPUT}
    
    """
    return base_system_msg, base_user_prompt

### Step 2: Test your prompts


Run the following code to verify that your prompt produces the expected output:


In [19]:
# Unit test:
restaurant_paragraph = restaurant_list
base_system_msg, base_user_prompt = restaurant_data_structure_prompt_generation(restaurant_paragraph=restaurant_paragraph)

test_response = llm_model(system_msg=base_system_msg, prompt_txt=base_user_prompt)
print(test_response)

[  
  {  
    "name": "The Gilded Artichoke",  
    "location": "Silver Lake",  
    "type": "upscale bistro",  
    "food_style": "Farm-to-Table Californian",  
    "rating": 4.5,  
    "price_range": 3,  
    "signatures": [  
      "lavender-rubbed roasted chicken",  
      "delicate heirloom tomato tarts"  
    ],  
    "vibe": "bohemian chic",  
    "environment": "high-end greenhouse with reclaimed wood and floor-to-ceiling windows"  
  },  
  {  
    "name": "Mar de Cortez",  
    "location": "Santa Monica",  
    "type": "casual taqueria",  
    "food_style": "Baja-style seafood",  
    "rating": 4.2,  
    "price_range": 1,  
    "signatures": [  
      "beer-battered snapper tacos",  
      "zesty octopus ceviche"  
    ],  
    "vibe": "salt-air energy",  
    "environment": "premier spot for open-air dining near the pier"  
  },  
  {  
    "name": "Iron & Embers",  
    "location": "Arts District, DTLA",  
    "type": "industrial American steakhouse",  
    "food_style": "

### Step 3: Validate the LLM outputs


LLMs are not always perfectly reliable and may produce errors, even when strict rules are specified in the prompt. To ensure that the generated outputs strictly conform to the required JSON format, you will learn how to define a function that formally validates the LLM output. 

Here, you are provided with the function and the test. You don't have to implement anything. Run the following code to validate the test response you obtained in Step 2.


In [20]:
# Validation
from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional

### 3.1. Define the schema
class Restaurant(BaseModel):
    name: str
    location: str
    type: str
    food_style: str
    rating: Optional[float] = None
    price_range: Optional[int] = None
    signatures: List[str] = Field(default_factory=list)
    vibe: Optional[str] = None
    environment: str
    shortcomings: List[str] = Field(default_factory=list)


### 3.2. Use the validation method to validate the test_response from the unit test
try:
    restaurant_data = Restaurant.model_validate_json(test_response)
    print(f"Success! Validated: {restaurant_data.name}")
except ValidationError as e:
    print(f"Validation failed: {e.json()}")

Validation failed: [{"type":"json_invalid","loc":[],"msg":"Invalid JSON: EOF while parsing a value at line 115 column 16","input":"[  \n  {  \n    \"name\": \"The Gilded Artichoke\",  \n    \"location\": \"Silver Lake\",  \n    \"type\": \"upscale bistro\",  \n    \"food_style\": \"Farm-to-Table Californian\",  \n    \"rating\": 4.5,  \n    \"price_range\": 3,  \n    \"signatures\": [  \n      \"lavender-rubbed roasted chicken\",  \n      \"delicate heirloom tomato tarts\"  \n    ],  \n    \"vibe\": \"bohemian chic\",  \n    \"environment\": \"high-end greenhouse with reclaimed wood and floor-to-ceiling windows\"  \n  },  \n  {  \n    \"name\": \"Mar de Cortez\",  \n    \"location\": \"Santa Monica\",  \n    \"type\": \"casual taqueria\",  \n    \"food_style\": \"Baja-style seafood\",  \n    \"rating\": 4.2,  \n    \"price_range\": 1,  \n    \"signatures\": [  \n      \"beer-battered snapper tacos\",  \n      \"zesty octopus ceviche\"  \n    ],  \n    \"vibe\": \"salt-air energy\",  \n

Don't worry if your validation failed; we'll address that in the next exercise. 


----


## **Exercise 4: Structure all the restaurant data**


### Step 1: Define prompts to have an LLM that auto repairs outputs to JSON format


As mentioned earlier, LLMs are not perfect. While you previously defined a schema to validate the output, validation alone does not fix incorrect results. To address this, introduce an additional LLM “expert” whose sole responsibility is to automatically repair and correct outputs that do not conform to the required JSON format.

In this step, you will define the prompt template function for this LLM. This function takes:
- `candidate_json_output`: The candidate json output
- `error_message`: The error message generated by the schema validation if a mistake is detected

In your prompts, you should tell LLM:
- The original wrong output by feeding `candidate_json_output`
- The guidance on correction by feeding `error_message`


In [21]:
def JSON_auto_repair_prompts(candidate_json_output, error_message):

    auto_repair_system_msg = """
    You are an expert JSON repair assistant.

    Your task is to repair invalid JSON and make it
    conform to the required restaurant data structure.

    Use the provided JSON candidate and validation error
    to identify and fix the problems.

    Do not invent missing information.
    Preserve all existing valid information.
    Return valid JSON only, without explanations or Markdown.
    """

    auto_repair_prompt = f"""
    Task:
    Repair the following JSON output based on the validation error.

    Candidate JSON:
    {candidate_json_output}

    Validation Error:
    {error_message}

    Expected JSON Structure:
    {EXAMPLE_OUTPUT}

    Instructions:
    - Fix JSON syntax errors.
    - Correct data types when the intended value is clear.
    - Follow the expected JSON structure.
    - Do not invent missing information.
    - Preserve existing valid values.
    - Return valid JSON only.
    """

    return auto_repair_system_msg, auto_repair_prompt

### Step 2: Run the for loop to go over all the restaurant data in the list (approximately 20 minutes)


You will structure each restaurant description paragraph into a JSON-formatted output by iterating through the dataset using a for loop.

Although using an LLM to auto repair the response format is not the best practice, here you are guaranteed that the generation and repair tasks here are simple enough for LLMs to accomplish.


In [36]:
structured_restaurant_lists = []
for i, restaurant_paragraph in enumerate(restaurant_list):
    print(restaurant_paragraph)
    system_msg = base_system_msg
    auto_repair_prompt = ""
    ### 2.1: Produce your initial output
    candidate_json_output = llm_model(
        system_msg=base_system_msg,
        prompt_txt=f"{base_user_prompt}\n\n{restaurant_paragraph}"
    )
    ### 2.2: Validation and Auto Correction loop
    retries = 0
    max_retries = 3
    skipped = False
    while True:
        try:
            # Validate the candidate JSON
            restaurant_data = Restaurant.model_validate_json(
                candidate_json_output
            )
            print("JSON is valid!")
            break
        except ValidationError as e:
            print("JSON has validation errors!")
            retries += 1
            if retries > max_retries:
                skipped = True
                break
            error_message = e.json()
            # Generate repair prompts
            system_msg, auto_repair_prompt = JSON_auto_repair_prompts(
                candidate_json_output,
                error_message
            )
            # Ask the LLM to repair the JSON
            candidate_json_output = llm_model(
                system_msg=system_msg,
                prompt_txt=auto_repair_prompt
            )
    ### 2.3: Append finalized response
    if not skipped:
        structured_restaurant_lists.append(
            restaurant_data.model_dump()
        )
    # A manual progress bar
    if (i + 1) % 20 == 0:
        print(f'{i+1} out of {len(restaurant_list)} is done')
# A final message
print('ALL DONE!!')

**The Gilded Artichoke** brings a **bohemian chic** energy to the hills of **Silver Lake**, operating as an **upscale bistro** that prioritizes **Farm-to-Table Californian** ingredients. The space feels like a high-end greenhouse with its reclaimed wood and floor-to-ceiling windows, perfectly complementing the **4.5/5** rating earned by its lavender-rubbed roasted chicken and delicate heirloom tomato tarts.  Price range: $$$$
JSON is valid!
Down in **Santa Monica**, **Mar de Cortez** serves as a **sun-drenched**, **casual taqueria** specializing in **Baja-style seafood**. With a **4.2/5** rating, it captures the salt-air energy of the coast through its signature beer-battered snapper tacos and zesty octopus ceviche, making it a premier spot for open-air dining near the pier. Price range: $
JSON has validation errors!
JSON is valid!
**Iron & Embers** anchors the **Arts District** in **DTLA** as a **moody**, **industrial American steakhouse**. This **fine dining** establishment earns a *

Take a screenshot of the Python code, clearly showing your implementation and the output from Step 2. Name the screenshot `M1L1_structure_for_loop.jpg`. 


### Step 3: Save the list to a JSON file


You are almost there! First, print the 50th item in your structured_restaurant_lists.


In [42]:
import json
print(structured_restaurant_lists[3])
# # 1. Prepare the restaurant data
structured_restaurant_lists_json = structured_restaurant_lists
print(structured_restaurant_lists_json[3])
# # 2. Assign itemId to each restaurant
for i, response in enumerate(structured_restaurant_lists_json):
    response["itemId"] = 1000001 + i

# 3. Save the data as a JSON file
filename = "structured_restaurant_data.json"

with open(filename, "w", encoding="utf-8") as f:
    json.dump(
        structured_restaurant_lists_json,
        f,
        ensure_ascii=False,
        indent=4
    )

print(f"Successfully saved {len(structured_restaurant_lists_json)} restaurants!")

# 4. Verify the saved JSON file
with open(filename, "r", encoding="utf-8") as f:
    loaded_data = json.load(f)

print(type(loaded_data))
print(loaded_data[1])

{'name': 'Sakura Garden', 'location': 'Little Tokyo', 'type': 'intimate Japanese Kaiseki-lite', 'food_style': 'seasonal perfection', 'rating': 4.9, 'price_range': 3, 'signatures': ['buttery bluefin tuna belly'], 'vibe': 'zen and minimalist', 'environment': 'a quiet escape focusing on seasonal perfection, where the bluefin tuna belly is served with a level of reverence.', 'shortcomings': [], 'itemId': 1000004}
{'name': 'Sakura Garden', 'location': 'Little Tokyo', 'type': 'intimate Japanese Kaiseki-lite', 'food_style': 'seasonal perfection', 'rating': 4.9, 'price_range': 3, 'signatures': ['buttery bluefin tuna belly'], 'vibe': 'zen and minimalist', 'environment': 'a quiet escape focusing on seasonal perfection, where the bluefin tuna belly is served with a level of reverence.', 'shortcomings': [], 'itemId': 1000004}
Successfully saved 210 restaurants!
<class 'list'>
{'name': 'Mar de Cortez', 'location': 'Santa Monica', 'type': 'casual taqueria', 'food_style': 'Baja-style seafood', 'ratin

Then, save your processed data to your folder by running the following code!


In [43]:
structured_restaurant_lists_json = [json.loads(json.dumps((response))) for response in structured_restaurant_lists]
#For each item in the restaurant list, assign it with an itemId to be consistent with the one in the user review data:
for i, response in enumerate(structured_restaurant_lists_json):
    response['itemId'] = 1000001 + i
    structured_restaurant_lists_json[i] = response
    
filename = 'structured_restaurant_data.json'
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(structured_restaurant_lists_json, f, indent=4)

You can download the saved JSON file to your local folder. Be sure to keep it in a safe location, as you will need it for future assignments.


----


## **Conclusion**


You have successfully applied GenAI tools to transform the unstructured text data into a well-structured JSON file!


## Authors


[Jianping (Mike) Ye](https://www.linkedin.com/in/jianping-ye/)


<!--## Changelog
|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2026-02-02|1|Jianping Ye|Create lab|
|2026-02-09|2|Jojy John|ID Review|-->


Copyright IBM Corporation. All rights reserved.
